# PhoBERT-CRF & XLM-R-large-CRF — NewsMNER (text-only)

Notebook này train và eval 2 model text-only NER trên NewsMNER, dùng chung script
`train/text_only/train_xlmr_crf.py` sẵn có trong repo (encoder + linear + CRF), chỉ khác `--bert_model`:

- **PhoBERT-CRF**: `vinai/phobert-base-v2`
- **XLM-R-large-CRF**: `xlm-roberta-large`

## Trước khi chạy

1. **Settings > Accelerator**: bật GPU (T4 x2 hoặc P100).
2. **Settings > Internet**: bật ON (cần để `pip install` và `git clone`).
3. **Add Data**: upload NewsMNER (định dạng CoNLL, tab-separated `token\tlabel`, câu cách nhau bằng dòng trống)
   gồm 3 file `train.txt` / `dev.txt` / `test.txt` thành 1 Kaggle Dataset, attach vào notebook, rồi sửa
   `DATA_DIR` ở cell "Dữ liệu" bên dưới cho khớp path `/kaggle/input/...`.
4. **QUAN TRỌNG**: `train/text_only/train_xlmr_crf.py`, `modules/model_architecture/XLMR_CRF_TextOnly.py`,
   `modules/datasets/dataset_text_only.py` hiện đang **chưa được commit/push** lên GitHub (untracked trong repo local).
   Trước khi notebook này clone được code, hãy `git add`, `git commit`, `git push` các file đó lên
   `https://github.com/CongMoc/MNER.git`. Nếu không, cell clone bên dưới sẽ thiếu code và các cell training phía sau
   sẽ báo `ModuleNotFoundError`.


## 1. Cài đặt phụ thuộc

CRF được vendor sẵn trong `modules/model_architecture/torchcrf.py` nên không cần cài `pytorch-crf`.
Chỉ cần thêm `pytorch_pretrained_bert` (cho `BertAdam`) và `seqeval`; `--no-deps` để tránh pip cài đè lại torch/CUDA đã có sẵn trên Kaggle.

In [ ]:
!pip install -q seqeval
!pip install -q --no-deps "pytorch_pretrained_bert==0.4.0"
!pip install -q -U transformers


## 2. Lấy code repo

Clone shallow (`--depth 1`) từ GitHub. Nếu bạn thay bằng Kaggle Dataset chứa code repo thay vì git clone, sửa cell này để copy từ `/kaggle/input/<your-code-dataset>` thay vì `git clone`.

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/CongMoc/MNER.git"
REPO_DIR = "/kaggle/working/MNER-Vietnamese"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo already present at", REPO_DIR)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


In [ ]:
# Sanity check: các file cần cho text-only CRF training phải có mặt sau khi clone
required = [
    "train/text_only/train_xlmr_crf.py",
    "modules/model_architecture/XLMR_CRF_TextOnly.py",
    "modules/datasets/dataset_text_only.py",
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        f"Thieu file: {missing}. Cac file nay chua duoc push len GitHub - "
        "hay commit/push roi xoa {REPO_DIR} va chay lai cell clone.".format(REPO_DIR=REPO_DIR)
    )
print("OK - da co du code can thiet.")


## 3. Dữ liệu

Sửa `DATA_DIR` trỏ đúng vào Kaggle Dataset NewsMNER (đã convert sang CoNLL) mà bạn attach vào notebook.

In [ ]:
DATA_DIR = "/kaggle/input/newsmner-textonly/converted"  # <-- EDIT: sua theo dataset ban upload

for fname in ("train.txt", "dev.txt", "test.txt"):
    fpath = os.path.join(DATA_DIR, fname)
    assert os.path.exists(fpath), (
        f"Thieu {fpath} -- upload NewsMNER (converted CoNLL) thanh Kaggle Dataset roi sua DATA_DIR."
    )
print("Data OK:", os.listdir(DATA_DIR))


In [ ]:
# Label set NewsMNER (giong scripts/server_ops/run_text_only_newsmner.sh)
os.environ["LABELS"] = "B-DATE,B-LOC,B-MISC,B-NUM,B-ORG,B-PER,I-DATE,I-LOC,I-MISC,I-NUM,I-ORG,I-PER,O,X,<s>,</s>"


## 4. Hàm chạy training

Gọi thẳng `train/text_only/train_xlmr_crf.py` qua `subprocess` với các hyperparameter mặc định giống
`scripts/server_ops/run_text_only_newsmner.sh` (10 epoch, batch 32, lr 2.2e-5, warmup 0.4, max_seq_length 256, seed 37).

**Lưu ý OOM**: GPU Kaggle (T4 16GB / P100 16GB) chạy `xlm-roberta-large` + seq_len 256 + batch 32 có thể tràn bộ nhớ.
Nếu gặp OOM, giảm `train_batch_size` (vd 16 hoặc 8) và tăng `gradient_accumulation_steps` tương ứng
(vd 2 hoặc 4) để giữ effective batch size 32.

In [ ]:
import sys, time

def run_training(bert_model, output_dir, train_batch_size=32, num_train_epochs=10.0,
                  learning_rate=2.2e-5, max_seq_length=256, warmup_proportion=0.4,
                  gradient_accumulation_steps=1, seed=37, task_name="newsmner"):
    cmd = [
        sys.executable, "train/text_only/train_xlmr_crf.py",
        "--do_train", "--do_eval",
        "--bert_model", bert_model,
        "--data_dir", DATA_DIR,
        "--output_dir", output_dir,
        "--task_name", task_name,
        "--num_train_epochs", str(num_train_epochs),
        "--train_batch_size", str(train_batch_size),
        "--gradient_accumulation_steps", str(gradient_accumulation_steps),
        "--learning_rate", str(learning_rate),
        "--warmup_proportion", str(warmup_proportion),
        "--max_seq_length", str(max_seq_length),
        "--cache_dir", "/kaggle/working/cache",
        "--seed", str(seed),
    ]
    print("Running:", " ".join(cmd))
    t0 = time.time()
    subprocess.run(cmd, check=True)
    print(f"Done in {(time.time() - t0) / 60:.1f} min -> {output_dir}")


## 5. PhoBERT-CRF

In [ ]:
run_training(
    bert_model="vinai/phobert-base-v2",
    output_dir="/kaggle/working/output/newsmner_phobert_textonly",
)


## 6. XLM-R-large-CRF

In [ ]:
run_training(
    bert_model="xlm-roberta-large",
    output_dir="/kaggle/working/output/newsmner_xlmr_large_textonly",
    train_batch_size=16,
    gradient_accumulation_steps=2,  # effective batch size 32, tranh OOM tren GPU 16GB
)


## 7. Kết quả

In [ ]:
import glob

for f in sorted(glob.glob("/kaggle/working/output/*/eval_results.txt")):
    print("=" * 80)
    print(f)
    print(open(f).read())


In [ ]:
!cd /kaggle/working && zip -rq output.zip output && echo "Saved /kaggle/working/output.zip"
